In [2]:
!pip install ucimlrepo

In [1]:


!pip install ucimlrepo
import sys
import subprocess

# Auto-install attempt if missing
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    print("ucimlrepo not found. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ucimlrepo"])
    from ucimlrepo import fetch_ucirepo
    print("Installation successful.")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# --- Step 1: Load and Clean Dataset ---
print("Fetching Adult Dataset (this may take a moment)...")
adult = fetch_ucirepo(id=2) 
X = adult.data.features
y = adult.data.targets

# CLEANING: The Adult dataset is notorious for trailing dots in the test set labels
y_cleaned = y.iloc[:, 0].astype(str).str.strip().str.replace('.', '', regex=False)

# Convert categorical features to dummy variables
X = pd.get_dummies(X).astype(float)
y = LabelEncoder().fit_transform(y_cleaned)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling (Crucial for SVM-based extraction)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Step 2: Extract Support Vectors (SVs) ---
print("Extracting Support Vectors (SVs)...")
# We use a 10k sample to find the boundary vectors quickly
svm_model = SVC(kernel='rbf', C=1.0)
svm_model.fit(X_train_scaled[:10000], y_train[:10000]) 

sv_indices = svm_model.support_
X_sv = X_train_scaled[sv_indices]
y_sv = y_train[sv_indices]
print(f"-> SVs Extracted: {len(X_sv)}")

# --- Step 3: Adversarial Margin Augmentation (AMA) ---
def apply_ama(X_sv, noise_level=0.01):
    # Generates synthetic data points right on the margin
    noise = np.random.normal(0, noise_level, X_sv.shape)
    return X_sv + noise

X_sv_syn = apply_ama(X_sv) 
X_combined_sv = np.vstack([X_sv, X_sv_syn])
y_combined_sv = np.concatenate([y_sv, y_sv])

# --- Step 4: Smart Sampling on Remaining Data ---
mask = np.ones(len(X_train_scaled), dtype=bool)
# Use the indices from our 10k subset to avoid overlap
used_indices = np.arange(10000)[sv_indices]
mask[used_indices] = False

X_remaining = X_train_scaled[mask]
y_remaining = y_train[mask]

# 5% sample for global structure context
sample_size = int(len(X_remaining) * 0.05)
idx = np.random.choice(len(X_remaining), sample_size, replace=False)
X_smart_sample = X_remaining[idx]
y_smart_sample = y_remaining[idx]

# --- Step 5: Merge & Final Distillation ---
X_distilled = np.vstack([X_combined_sv, X_smart_sample])
y_distilled = np.concatenate([y_combined_sv, y_smart_sample])
print(f"-> Distilled Dataset Size: {len(X_distilled)} (Original: {len(X_train)})")

# --- Step 6: Train Final Ensemble Models ---
print("\nTraining Final Models on Distilled Data...")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_distilled, y_distilled)
y_pred_rf = rf.predict(X_test_scaled)

# XGBoost
xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_distilled, y_distilled)
y_pred_xgb = xgb.predict(X_test_scaled)

# --- Results ---
print("\n" + "="*40)
print(f"{'MODEL':<20} | {'ACCURACY':<10}")
print("-" * 40)
print(f"{'Random Forest':<20} | {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"{'XGBoost':<20} | {accuracy_score(y_test, y_pred_xgb):.4f}")
print("="*40)


Fetching Adult Dataset (this may take a moment)...
Extracting Support Vectors (SVs)...
-> SVs Extracted: 4083
-> Distilled Dataset Size: 9915 (Original: 39073)

Training Final Models on Distilled Data...

MODEL                | ACCURACY  
----------------------------------------
Random Forest        | 0.8352
XGBoost              | 0.8540
